In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import col, to_timestamp, upper, current_timestamp
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "proyecto_olist")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_customers = spark.table(f"{catalogo}.{esquema_source}.customers")

In [0]:
df_customers_clean = (
    df_customers
    .select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_zip_code_prefix").alias("customer_zip_code_prefix"),
        upper(col("customer_city")).alias("customer_city"),
        col("customer_state").alias("customer_state")
    )
    .withColumn("silver_insert_date", current_timestamp())
)

In [0]:
df_customers_clean.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.customers")

In [0]:
print("Tabla silver.customers procesada.")